# Wavebit Tutorial 2
## Gradient and Optimization with [JAX](https://jax.readthedocs.io) and [Optax](https://optax.readthedocs.io/)

In [1]:
import functools

import optax

import jax.numpy as jnp

import wavebit

In [2]:
backend = 1  # JAX

# common
psi = jnp.pi / 2  # pure s (TE-like); psi is the linear angle

top = 1.0  # superstrate: relative permittivity. 1.0 is vacuum
bot = 1.0  # substrate

theta = 0 * jnp.pi / 180  # angle of incidence
phi = 0 * jnp.pi / 180  # angle of rotation

wavelength = 900

thickness = [500., 1000.]  # thickness of each layer, from top to bottom.
period = [1000.]  # length of the unit cell. Here it's 1D.

max_order = [10]

complex_dtype = jnp.complex128

In [3]:
structure_1d_m = jnp.array([
    [[0, 0, 0, 1, 1, 1, 1, 0, 0, 0, ]],
    [[1, 1, 1, 1, 0, 1, 1, 1, 1, 1, ]],
    ]) * 24. + 1.  # relative permittivity: 1 (vacuum) and 25

# The design variable is the permittivity itself, one number per pixel. That is what a
# raster is for -- it is a sampled field rather than an assignment, so every pixel is a
# knob an optimizer can turn. A gradient taken here is d(loss)/d(eps); it was
# d(loss)/d(n) before the material convention changed, and the two differ by the chain
# factor 2n, so numbers from an older run of this notebook do not carry over.

## 2.1 Gradient

Gradient can be calculated with the help of `jax.value_and_grad` function.
Read this for further information: [Automatic Differentiation](https://jax.readthedocs.io/en/latest/jax-101/04-advanced-autodiff.html)

Optax is used for optimization. Like PyTorch, Optax also provides various loss functions and optimizers so users easily can utilize well-established implementations. Refer this tutorial: [Learn Optax](https://optax.readthedocs.io/en/latest/optax-101.html)


Gradient can be utilized to solve optimization problems. Here are examples that show couple of ways to get gradient or optimized values with or without predefined functions of wavebit.

### 2.1.1 Examples

In [4]:
class Loss:
    def __call__(self, result, *args, **kwargs):
        given, res_te, res_ti = result.given, result.s_inc, result.p_inc
        T = given.T
        center = [a // 2 for a in T.shape]
        res = T[center[0], center[1]+1]

        return res


loss_fn = Loss()

In [5]:
solver = wavebit.rcwa(backend=backend, psi=psi, top=top, bot=bot, theta=theta, phi=phi, max_order=max_order, wavelength=wavelength, period=period, structure=structure_1d_m, thickness=thickness, complex_dtype=complex_dtype)

parameters = ['structure', 'thickness']
# Freeze the discrete geometry decisions once, so jax.jit has nothing left to
# trace through. This sampled path no longer needs it, but NVF and the integrated path do.
geometry = solver.freeze_geometry()
forward = functools.partial(solver.solve, geometry=geometry)

# case 1: Gradient
grad = solver.grad(parameters, forward, loss_fn)

print('structure gradient:')
print(grad['structure'])
print('thickness gradient:')
print(grad['thickness'])

structure gradient:
[[[-0.02569226 -0.01271391 -0.00366063  0.00786614 -0.00187536
    0.0094884   0.00879497 -0.00063303 -0.01831762 -0.02404682]]

 [[-0.01782688 -0.00829977 -0.02213984 -0.01918077  0.04603974
    0.00588386 -0.00420576 -0.01347758 -0.02978962  0.01309464]]]
thickness gradient:
[ 0.00222953 -0.00675975]


## 2.2 Optimization

### 2.2.1 Examples

Example 1

In [6]:
solver = wavebit.rcwa(backend=backend, psi=psi, top=top, bot=bot, theta=theta, phi=phi, max_order=max_order, wavelength=wavelength, period=period, structure=structure_1d_m, thickness=thickness, complex_dtype=complex_dtype)

parameters = ['structure', 'thickness']
# Freeze the discrete geometry decisions once, so jax.jit has nothing left to
# trace through. This sampled path no longer needs it, but NVF and integrated do.
geometry = solver.freeze_geometry()
forward = functools.partial(solver.solve, geometry=geometry)

# case 2: SGD
optimizer = optax.sgd(learning_rate=1e-2, momentum=0.9)
res = solver.fit(parameters, forward, loss_fn, optimizer, iteration=3)

print('structure final:')
print(res['structure'])
print('thickness final:')
print(res['thickness'])

  0%|          | 0/3 [00:00<?, ?it/s]

 33%|███▎      | 1/3 [00:01<00:03,  1.66s/it]

100%|██████████| 3/3 [00:01<00:00,  1.80it/s]

structure final:
[[[ 1.00144239  1.00071383  1.00020551 24.99955973 25.00010559
   24.99946894 24.99950787  1.00003555  1.00102798  1.00135029]]

 [[25.00100062 25.00046685 25.00124131 25.00107587  0.99742322
   24.99967126 25.00023718 25.00075549 25.00167066 24.99926687]]]
thickness final:
[ 499.99987521 1000.00037917]
